# Práctica 14 — Proyecto integrador

**Curso:** Estadística Aplicada con Python · **Bloque 5:** Cierre · **Tiempo estimado:** 3-4 horas de trabajo autónomo (un caso)
**Clase asociada:** `notebooks_clase/clase14_proyecto_integrador.ipynb` · **Capítulo:** `manual/14_proyecto_integrador.md`

## Cómo funciona esta práctica

- Hay **tres casos**: **A** (salud: diabetes), **B** (comercio electrónico con un test A/B) y **C** (precios de viviendas en California). **Elige UNO y complétalo**; si te sobra tiempo, haz los otros.
- Cada caso tiene **3 pasos** con una celda `# TODO` y una **comprobación automática** (`assert`): (1) auditoría y exploración, (2) modelo o test adecuado con diagnóstico, (3) un diccionario `resultado_X` con las cifras clave.
- Al terminar, la **sfida final** te pide escribir el informe de una página (`informe_final`). La comprobación solo verifica lo mínimo (secciones, cifras y límites); la calidad la valoras tú con la **rúbrica** de la última celda.
- Ejecuta las celdas de **setup y de datos** y después **solo las celdas de tu caso** (las de los otros casos darán error si no las completas: es normal).
- Las **soluciones** de los tres casos están en una sección aparte al final. **No las abras antes de haberlo intentado.**
- Recuerda: A y C son datos **observacionales** (nada de "causa"); en B la causalidad solo es defendible si la aleatorización aguanta: por eso se comprueba.

---
## Celda 0 — Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings("ignore")

rng = np.random.default_rng(42)          # semilla fija: todos obtenemos los mismos resultados
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep
from statsmodels.stats.multitest import multipletests
import re
import unicodedata

def vif_de(X):
    """VIF de cada columna de X (DataFrame numérico sin constante). Devuelve un diccionario {columna: VIF}."""
    Xc = sm.add_constant(X.astype(float))
    return {col: variance_inflation_factor(Xc.values, i) for i, col in enumerate(Xc.columns) if col != "const"}

print("Librerías cargadas correctamente.")

---
## Datos de los tres casos

Se cargan los tres conjuntos de datos (A y C: CSV local → `scikit-learn` → sintético; B: generado con semilla fija).

In [ ]:
def generar_diabetes_sintetico(n=442, semilla=2026):
    """Dataset SINTÉTICO con las mismas columnas que `diabetes` (ejemplo ilustrativo, NO son pacientes reales)."""
    r = np.random.default_rng(semilla)
    age = np.round(np.clip(r.normal(48.5, 13, n), 19, 79))
    sex = r.choice([1.0, 2.0], size=n)
    bmi = np.round(np.clip(r.normal(26.4, 4.4, n), 18, 42), 1)
    bp = np.round(np.clip(70 + 1.0 * (bmi - 26) + 0.3 * age + r.normal(0, 11, n), 62, 133))
    s1 = np.round(np.clip(r.normal(189, 34, n), 97, 301))
    s2 = np.round(np.clip(0.75 * (s1 - 189) + 115 + r.normal(0, 6, n), 42, 242), 1)   # muy correlacionada con s1
    s3 = np.round(np.clip(50 - 0.15 * (s2 - 115) + r.normal(0, 11, n), 22, 99))
    s4 = np.round(np.clip(4.07 + 0.02 * (s1 - 189) - 0.05 * (s3 - 50) + r.normal(0, 0.6, n), 2, 9), 2)
    s5 = np.round(np.clip(4.6 + 0.004 * (s1 - 189) - 0.008 * (s3 - 50) + r.normal(0, 0.3, n), 3.2, 6.1), 4)
    s6 = np.round(np.clip(r.normal(91, 11.5, n), 58, 124))
    target = np.round(np.clip(152 + 5.5 * (bmi - 26.4) + 1.1 * (bp - 94.6) + 40 * (s5 - 4.6) - 1.0 * (s3 - 50)
                              - 20 * (sex - 1.5) + r.normal(0, 55, n), 25, 346))
    return pd.DataFrame({"age": age, "sex": sex, "bmi": bmi, "bp": bp, "s1": s1, "s2": s2, "s3": s3,
                         "s4": s4, "s5": s5, "s6": s6, "target": target})


def cargar_diabetes():
    """Devuelve (DataFrame, origen): CSV local, si no scikit-learn (valores originales), si no datos sintéticos."""
    for ruta in ("../datos/diabetes.csv", "datos/diabetes.csv", "diabetes.csv"):
        if os.path.exists(ruta):
            return pd.read_csv(ruta), "CSV local"
    try:
        from sklearn.datasets import load_diabetes
        return load_diabetes(as_frame=True, scaled=False).frame, "scikit-learn"
    except Exception:
        return generar_diabetes_sintetico(), "sintético"

def generar_california_sintetico(n=5000, semilla=42):
    """Dataset SINTÉTICO con las mismas columnas que California housing (ejemplo ilustrativo: las relaciones están inventadas)."""
    g = np.random.default_rng(semilla)
    MedInc = np.clip(g.lognormal(1.25, 0.5, n), 0.5, 15)
    HouseAge = g.integers(1, 53, n).astype(float)
    AveRooms = np.clip(3.0 + 0.35 * MedInc + g.normal(0, 0.7, n), 2, 9)
    AveBedrms = np.clip(0.2 * AveRooms + g.normal(0, 0.05, n), 0.6, 2)
    Population = np.round(g.lognormal(7.0, 0.6, n))
    AveOccup = np.clip(g.lognormal(1.05, 0.25, n), 1.2, 5.5)
    Latitude = np.round(g.uniform(32.5, 42.0, n), 2)
    Longitude = np.round(g.uniform(-124.3, -114.5, n), 2)
    valor = (0.5 + 0.42 * MedInc + 0.012 * HouseAge - 0.15 * AveRooms - 0.3 * AveOccup
             + 0.05 * (37 - Latitude) + g.normal(0, 0.6, n))
    MedHouseVal = np.clip(valor, 0.15, 5.00001)
    return pd.DataFrame({"MedInc": MedInc, "HouseAge": HouseAge, "AveRooms": AveRooms, "AveBedrms": AveBedrms,
                         "Population": Population, "AveOccup": AveOccup, "Latitude": Latitude,
                         "Longitude": Longitude, "MedHouseVal": MedHouseVal})


def cargar_california():
    """Devuelve (DataFrame, origen): CSV local, si no scikit-learn, si no datos sintéticos."""
    for ruta in ("../datos/california_housing.csv", "datos/california_housing.csv", "california_housing.csv"):
        if os.path.exists(ruta):
            return pd.read_csv(ruta), "CSV local"
    try:
        from sklearn.datasets import fetch_california_housing
        datos = fetch_california_housing(as_frame=True).frame
        return datos.sample(n=5000, random_state=42).reset_index(drop=True), "scikit-learn"
    except Exception:
        return generar_california_sintetico(), "sintético"

def generar_ecommerce(n=12000, semilla=1):
    """Genera el caso B: un test A/B de comercio electrónico con datos SINTÉTICOS (ejemplo ilustrativo).
    Efecto verdadero de B sobre la conversión: positivo y moderado (+0,10 en log-odds en escritorio, +0,30 en móvil).
    Trampa deliberada: la asignación NO es 50/50 (en móvil, 54 % va a B) -> SRM y desequilibrio de dispositivos.
    Los ingresos son 0 si no convierte y lognormales (cola larga) si convierte."""
    r = np.random.default_rng(semilla)
    dispositivo = np.where(r.random(n) < 0.55, "movil", "escritorio")
    canal = r.choice(["organico", "pago", "email"], size=n, p=[0.5, 0.3, 0.2])
    sesiones_previas = r.poisson(2.0, n)
    prob_b = np.where(dispositivo == "movil", 0.54, 0.50)
    grupo = np.where(r.random(n) < prob_b, "B", "A")
    logit = (-2.7 + np.where(dispositivo == "movil", -0.25, 0.0)
             + np.select([canal == "pago", canal == "email"], [-0.30, 0.45], 0.0)
             + 0.10 * sesiones_previas
             + (grupo == "B") * np.where(dispositivo == "movil", 0.30, 0.10))
    convirtio = (r.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    ingresos = np.where(convirtio == 1, np.round(r.lognormal(3.6, 0.9, n), 2), 0.0)
    return pd.DataFrame({"id_usuario": np.arange(1, n + 1), "grupo": grupo, "dispositivo": dispositivo,
                         "canal": canal, "sesiones_previas": sesiones_previas, "convirtio": convirtio,
                         "ingresos": ingresos})


diab, origen_a = cargar_diabetes()
cali, origen_c = cargar_california()
ecom = generar_ecommerce()
print("Caso A (diabetes):", diab.shape, "— origen:", origen_a)
print("Caso B (comercio electrónico, sintético):", ecom.shape)
print("Caso C (viviendas de California):", cali.shape, "— origen:", origen_c)

---
# Caso A — Salud: progresión de la diabetes

**Contexto.** 442 pacientes con diabetes. Para cada uno: `age`, `sex` (codificado 1/2; el conjunto de datos no dice cuál es cuál, así que **no interpretes el signo** como "hombre/mujer"), `bmi` (índice de masa corporal), `bp` (presión arterial media), seis análisis de sangre `s1`-`s6` y `target`, una medida cuantitativa de la **progresión de la enfermedad un año después** (valores originales, sin escalar). Son datos **observacionales**.

**Pregunta de negocio.** Un equipo clínico quiere saber qué características basales se asocian con una mayor progresión, para priorizar el seguimiento.

**Ruta.** Auditoría (clases 2-3) → correlaciones y multicolinealidad (clase 12: VIF) → regresión múltiple con errores robustos HC3 (clase 10) → efecto del `bmi` con IC.
**Trampas:** `s1`, `s2` (y `s3`) miden cosas parecidas, con VIF muy altos; con n = 442 no se puede afirmar causa ni predecir bien sin validación.

### Caso A · Paso 1 — Auditoría y exploración

Calcula sobre `diab`: `n_a` (filas), `n_faltantes_a` (total de valores ausentes), `n_duplicados_a` (filas duplicadas), `r_bmi` (correlación de Pearson entre `bmi` y `target`), `var_max_corr` (nombre de la variable, sin contar `target`, con mayor correlación **absoluta** con `target`) y `r_s1_s2` (correlación entre `s1` y `s2`: la señal de multicolinealidad).

In [ ]:
n_a = None              # TODO
n_faltantes_a = None    # TODO
n_duplicados_a = None   # TODO
r_bmi = None            # TODO
var_max_corr = None     # TODO: pista: diab.drop(columns="target").corrwith(...)
r_s1_s2 = None          # TODO

# Comprobación automática
assert all(v is not None for v in (n_a, n_faltantes_a, n_duplicados_a, r_bmi, var_max_corr, r_s1_s2)), "Completa las líneas marcadas con TODO"
assert n_a == len(diab), "n_a = len(diab)"
assert n_faltantes_a == int(diab.isna().sum().sum()), "suma de todos los ausentes: diab.isna().sum().sum()"
assert n_duplicados_a == int(diab.duplicated().sum()), "diab.duplicated().sum()"
assert np.isclose(r_bmi, diab["bmi"].corr(diab["target"])), "r de Pearson entre bmi y target"
assert var_max_corr == diab.drop(columns="target").corrwith(diab["target"]).abs().idxmax(), "usa el valor absoluto de la correlación y idxmax()"
assert np.isclose(r_s1_s2, diab["s1"].corr(diab["s2"])), "r de Pearson entre s1 y s2"
print(f"n = {n_a}, ausentes = {n_faltantes_a}, duplicados = {n_duplicados_a}, r(bmi, target) = {r_bmi:.3f}, mayor |r|: {var_max_corr}, r(s1, s2) = {r_s1_s2:.3f}")
print("Caso A, paso 1 correcto ✔")

### Caso A · Paso 2 — Multicolinealidad y modelo con errores robustos

Con la matriz `X_a` (todas las explicativas, `sex` como 0/1) calcula `vif_a` (diccionario de VIF, con la función `vif_de` del setup), `vif_max_a` (el mayor VIF) y `variables_vif_alto` (lista **ordenada** de variables con VIF > 10). Después ajusta `modelo_a`: la regresión de `target` sobre `age`, `sex` (categórica), `bmi`, `bp`, `s3`, `s5` y `s6` (sin `s1`, `s2` y `s4`, que son redundantes) con **errores estándar robustos HC3**.

In [ ]:
X_a = diab.drop(columns="target").copy()
X_a["sex"] = (X_a["sex"] == 2).astype(float)     # 0/1: solo para poder calcular el VIF

vif_a = None                # TODO
vif_max_a = None            # TODO
variables_vif_alto = None   # TODO
modelo_a = None             # TODO: smf.ols("target ~ age + C(sex) + ...", data=diab).fit(cov_type="HC3")

# Comprobación automática
assert all(v is not None for v in (vif_a, vif_max_a, variables_vif_alto, modelo_a)), "Completa las líneas marcadas con TODO"
_Xn = X_a.astype(float).values
_ref = dict(zip(X_a.columns, np.diag(np.linalg.inv(np.corrcoef(_Xn.T)))))      # VIF = diagonal de la inversa de la matriz de correlaciones
assert set(vif_a) == set(X_a.columns) and all(np.isclose(vif_a[c], _ref[c], rtol=1e-6) for c in _ref), "vif_a = vif_de(X_a): un VIF por cada columna de X_a"
assert np.isclose(vif_max_a, max(_ref.values())), "vif_max_a = max(vif_a.values())"
assert variables_vif_alto == sorted(c for c, v in _ref.items() if v > 10), "sorted([c for c, v in vif_a.items() if v > 10])"
assert getattr(modelo_a, "cov_type", None) == "HC3", "falta .fit(cov_type='HC3')"
assert set(modelo_a.params.index) == {"Intercept", "C(sex)[T.2.0]", "age", "bmi", "bp", "s3", "s5", "s6"}, "fórmula: target ~ age + C(sex) + bmi + bp + s3 + s5 + s6"
print(f"VIF máximo (modelo completo) = {vif_max_a:.1f}; variables con VIF > 10:", variables_vif_alto)
print("Coeficiente de bmi (HC3):", round(modelo_a.params["bmi"], 3))
print("Caso A, paso 2 correcto ✔")

### Caso A · Paso 3 — Resultado con incertidumbre

Con `modelo_a` construye el diccionario `resultado_a` con las claves: `n` (observaciones del modelo), `coef_bmi` (coeficiente de `bmi`: cambio esperado en `target` por cada unidad de `bmi`, a igualdad de lo demás), `ic_inf` e `ic_sup` (IC 95 % con HC3), `r2_adj` (R² corregido), `vif_max` (mayor VIF **del modelo reducido**; calcúlalo con `vif_de` sobre `age`, `sex`, `bmi`, `bp`, `s3`, `s5`, `s6`) y `causal` (`True` o `False`: ¿autoriza estos datos a afirmar que reducir el `bmi` reducirá la progresión?).

In [ ]:
X_red = X_a[["age", "sex", "bmi", "bp", "s3", "s5", "s6"]]

resultado_a = {
    "n": None,          # TODO
    "coef_bmi": None,   # TODO
    "ic_inf": None,     # TODO
    "ic_sup": None,     # TODO
    "r2_adj": None,     # TODO
    "vif_max": None,    # TODO
    "causal": None,     # TODO
}

# Comprobación automática
assert all(v is not None for v in resultado_a.values()), "Completa las líneas marcadas con TODO"
_r = resultado_a
assert _r["n"] == len(diab), "n = número de observaciones del modelo (modelo_a.nobs)"
assert np.isclose(_r["coef_bmi"], modelo_a.params["bmi"]), "coef_bmi = modelo_a.params['bmi']"
_ic = modelo_a.conf_int().loc["bmi"]
assert np.isclose(_r["ic_inf"], _ic[0]) and np.isclose(_r["ic_sup"], _ic[1]), "IC 95 %: modelo_a.conf_int().loc['bmi']"
assert _r["ic_inf"] < _r["coef_bmi"] < _r["ic_sup"], "el IC debe contener la estimación"
assert np.isclose(_r["r2_adj"], smf.ols("target ~ age + C(sex) + bmi + bp + s3 + s5 + s6", data=diab).fit().rsquared_adj), "R² corregido: .rsquared_adj (el del modelo sin HC3 es el mismo valor)"
_v = np.diag(np.linalg.inv(np.corrcoef(X_red.astype(float).values.T))).max()
assert np.isclose(_r["vif_max"], _v, rtol=1e-6), "vif_max = max(vif_de(X_red).values())"
assert _r["causal"] is False, "son datos observacionales: no se puede afirmar causa"
print(f"Efecto de bmi: {_r['coef_bmi']:.2f} por unidad (IC 95 % HC3: {_r['ic_inf']:.2f} a {_r['ic_sup']:.2f}); R² corregido = {_r['r2_adj']:.3f}; VIF máximo = {_r['vif_max']:.2f}")
print("Caso A, paso 3 correcto ✔")

---
# Caso B — Comercio electrónico: ¿funciona la nueva página de pago?

**Contexto.** Datos **sintéticos** (*ejemplo ilustrativo*) de un test A/B con ~12 000 usuarios: `id_usuario`, `grupo` (`A` control, `B` nueva versión), `dispositivo` (`movil`/`escritorio`), `canal` (`organico`/`pago`/`email`), `sesiones_previas`, `convirtio` (0/1) e `ingresos` (0 si no convierte; con cola larga si convierte). La función `generar_ecommerce` está documentada: puedes leerla.

**Pregunta de negocio.** ¿Debe la empresa lanzar la versión B a todos los usuarios? ¿Y en qué segmentos?

**Ruta.** Auditoría (clases 2-3) → **controles de salud** del experimento: SRM y equilibrio de dispositivos (clase 13) → test de dos proporciones con IC de la diferencia (clases 7 y 13) → segmentos dispositivo × canal con **corrección por comparaciones múltiples** (clase 9).
**Trampas:** el reparto **no** es 50/50 (SRM) y sospecha de dónde viene; las medias de `ingresos` están dominadas por la cola larga; seis segmentos son seis contrastes.

### Caso B · Paso 1 — Auditoría y exploración

Sobre `ecom` calcula: `n_b` (filas), `n_faltantes_b` (total de ausentes), `n_duplicados_id` (identificadores `id_usuario` repetidos), `tasa_A` y `tasa_B` (tasa de conversión de cada grupo, **en %**) y `asimetria_ingresos` (asimetría, `scipy.stats.skew`, de `ingresos` **solo entre quienes convierten**: ¿por qué no sirve la media a secas?).

In [ ]:
n_b = None                 # TODO
n_faltantes_b = None       # TODO
n_duplicados_id = None     # TODO
tasa_A = None              # TODO: en %, no en proporción
tasa_B = None              # TODO
asimetria_ingresos = None  # TODO

# Comprobación automática
assert all(v is not None for v in (n_b, n_faltantes_b, n_duplicados_id, tasa_A, tasa_B, asimetria_ingresos)), "Completa las líneas marcadas con TODO"
assert n_b == len(ecom), "n_b = len(ecom)"
assert n_faltantes_b == int(ecom.isna().sum().sum()) and n_duplicados_id == int(ecom["id_usuario"].duplicated().sum()), "ecom.isna().sum().sum() y ecom['id_usuario'].duplicated().sum()"
assert np.isclose(tasa_A, ecom.loc[ecom["grupo"] == "A", "convirtio"].mean() * 100), "tasa_A: media de convirtio en el grupo A, multiplicada por 100"
assert np.isclose(tasa_B, ecom.loc[ecom["grupo"] == "B", "convirtio"].mean() * 100), "tasa_B: media de convirtio en el grupo B, multiplicada por 100"
assert np.isclose(asimetria_ingresos, stats.skew(ecom.loc[ecom["convirtio"] == 1, "ingresos"])), "stats.skew de ingresos filtrando convirtio == 1"
assert asimetria_ingresos > 1, "una asimetría > 1 indica cola larga: mejor la mediana o el bootstrap que la media"
print(f"n = {n_b}, ausentes = {n_faltantes_b}, ids repetidos = {n_duplicados_id}, conversión A = {tasa_A:.2f} %, conversión B = {tasa_B:.2f} %, asimetría de ingresos = {asimetria_ingresos:.2f}")
print("Caso B, paso 1 correcto ✔")

### Caso B · Paso 2 — Controles de salud y test de dos proporciones

Calcula: `tamanos_b` (diccionario con los usuarios de `A` y de `B`), `chi2_srm` y `p_srm` (chi-cuadrado de bondad de ajuste contra un reparto 50/50, con `scipy.stats.chisquare`), `srm_detectado` (`True` si `p_srm < 0.001`), `p_dispositivos` (p-value del chi-cuadrado de independencia entre `dispositivo` y `grupo`, con `chi2_contingency`), y el test de dos proporciones: `dif_pp` (tasa de B menos tasa de A, en puntos porcentuales), `ic_inf_pp` e `ic_sup_pp` (IC 95 % de la diferencia con `confint_proportions_2indep`, método `"newcomb"`, B primero, en pp) y `p_valor` (`proportions_ztest`, B primero).

In [ ]:
conv = ecom.groupby("grupo")["convirtio"].agg(["sum", "count"])     # conversiones y usuarios por grupo

tamanos_b = None       # TODO
chi2_srm = None        # TODO
p_srm = None           # TODO
srm_detectado = None   # TODO
p_dispositivos = None  # TODO
dif_pp = None          # TODO
ic_inf_pp = None       # TODO
ic_sup_pp = None       # TODO
p_valor = None         # TODO

# Comprobación automática
assert all(v is not None for v in (tamanos_b, chi2_srm, p_srm, srm_detectado, p_dispositivos, dif_pp, ic_inf_pp, ic_sup_pp, p_valor)), "Completa las líneas marcadas con TODO"
assert tamanos_b == {"A": int(conv.loc["A", "count"]), "B": int(conv.loc["B", "count"])}, "tamanos_b = {'A': usuarios de A, 'B': usuarios de B}"
_c2, _p = stats.chisquare([conv.loc["A", "count"], conv.loc["B", "count"]])
assert np.isclose(chi2_srm, _c2) and np.isclose(p_srm, _p), "stats.chisquare([n_A, n_B]) reparte por defecto a partes iguales"
assert srm_detectado == bool(_p < 0.001), "srm_detectado = p_srm < 0.001"
assert np.isclose(p_dispositivos, stats.chi2_contingency(pd.crosstab(ecom["dispositivo"], ecom["grupo"]))[1]), "chi2_contingency(pd.crosstab(dispositivo, grupo))[1]"
_d = (conv.loc["B", "sum"] / conv.loc["B", "count"] - conv.loc["A", "sum"] / conv.loc["A", "count"]) * 100
assert np.isclose(dif_pp, _d), "dif_pp = (tasa de B − tasa de A) · 100"
_lo, _hi = confint_proportions_2indep(conv.loc["B", "sum"], conv.loc["B", "count"], conv.loc["A", "sum"], conv.loc["A", "count"], method="newcomb")
assert np.isclose(ic_inf_pp, _lo * 100) and np.isclose(ic_sup_pp, _hi * 100), "confint_proportions_2indep(conv_B, n_B, conv_A, n_A, method='newcomb'), multiplicado por 100"
assert ic_inf_pp < dif_pp < ic_sup_pp, "el IC debe contener la estimación"
assert np.isclose(p_valor, proportions_ztest([conv.loc["B", "sum"], conv.loc["A", "sum"]], [conv.loc["B", "count"], conv.loc["A", "count"]])[1]), "proportions_ztest([conv_B, conv_A], [n_B, n_A])[1]"
print(f"Usuarios: {tamanos_b}  SRM: chi² = {chi2_srm:.1f}, p = {p_srm:.2g}  (¡alarma!)" if srm_detectado else f"SRM: p = {p_srm:.3f}")
print(f"Dispositivo × grupo: p = {p_dispositivos:.2g}")
print(f"Diferencia B − A = {dif_pp:.2f} pp (IC 95 %: {ic_inf_pp:.2f} a {ic_sup_pp:.2f}), p = {p_valor:.4f}")
print("Caso B, paso 2 correcto ✔")

### Caso B · Paso 3 — Segmentos con corrección y decisión

Para cada uno de los 6 segmentos `dispositivo × canal` (en el orden de `ecom.groupby(["dispositivo", "canal"])`) calcula el p-value del test de dos proporciones (B primero): `p_seg` (lista de 6 valores). Corrige con **Holm**: `p_holm` (`multipletests(p_seg, method="holm")[1]`). Cuenta `n_sig_sin` (segmentos con `p_seg < 0.05`) y `n_sig_holm` (con `p_holm < 0.05`). Por último, construye `resultado_b` con: `n`, `dif_pp`, `ic_inf_pp`, `ic_sup_pp`, `srm_p` (= `p_srm`), `n_seg_sig_holm` (= `n_sig_holm`) y `decision`: `"revisar_asignacion"` si hay SRM; si no, `"lanzar_B"` cuando el IC de la diferencia no incluye 0 y `"no_lanzar_B"` en caso contrario.

In [ ]:
p_seg = []
for _, seg in ecom.groupby(["dispositivo", "canal"]):
    pass    # TODO: conversiones y usuarios de A y de B en el segmento; añade el p-value a p_seg

p_holm = None       # TODO
n_sig_sin = None    # TODO
n_sig_holm = None   # TODO

resultado_b = {
    "n": None,               # TODO
    "dif_pp": None,          # TODO
    "ic_inf_pp": None,       # TODO
    "ic_sup_pp": None,       # TODO
    "srm_p": None,           # TODO
    "n_seg_sig_holm": None,  # TODO
    "decision": None,        # TODO
}

# Comprobación automática
assert p_holm is not None and n_sig_sin is not None and n_sig_holm is not None and all(v is not None for v in resultado_b.values()), "Completa las líneas marcadas con TODO"
_ps = []
for _, _s in ecom.groupby(["dispositivo", "canal"]):
    _t = _s.groupby("grupo")["convirtio"].agg(["sum", "count"])
    _ps.append(proportions_ztest([_t.loc["B", "sum"], _t.loc["A", "sum"]], [_t.loc["B", "count"], _t.loc["A", "count"]])[1])
assert len(p_seg) == 6 and np.allclose(p_seg, _ps), "6 p-values, uno por segmento, con B primero en proportions_ztest"
assert np.allclose(p_holm, multipletests(_ps, method="holm")[1]), "multipletests(p_seg, method='holm')[1]"
assert n_sig_sin == int(np.sum(np.array(_ps) < 0.05)) and n_sig_holm == int(np.sum(multipletests(_ps, method="holm")[1] < 0.05)), "cuenta cuántos p-values (sin corregir y corregidos) son < 0.05"
assert n_sig_holm <= n_sig_sin, "corregir nunca puede aumentar el número de significativos"
_r = resultado_b
assert _r["n"] == len(ecom) and np.isclose(_r["dif_pp"], dif_pp) and np.isclose(_r["ic_inf_pp"], ic_inf_pp) and np.isclose(_r["ic_sup_pp"], ic_sup_pp), "usa las cifras del paso 2"
assert np.isclose(_r["srm_p"], p_srm) and _r["n_seg_sig_holm"] == n_sig_holm, "srm_p = p_srm y n_seg_sig_holm = n_sig_holm"
assert _r["ic_inf_pp"] < _r["dif_pp"] < _r["ic_sup_pp"], "el IC debe contener la estimación"
assert _r["decision"] == ("revisar_asignacion" if p_srm < 0.001 else ("lanzar_B" if ic_inf_pp > 0 else "no_lanzar_B")), "si hay SRM la decisión es revisar la asignación, aunque el p-value sea pequeño"
print(f"Segmentos significativos sin corregir: {n_sig_sin}; con Holm: {n_sig_holm}")
print(f"Decisión: {_r['decision']}")
print("Caso B, paso 3 correcto ✔")

---
# Caso C — Inmobiliario: ¿de qué depende el valor de la vivienda?

**Contexto.** 5 000 distritos censales de California (los datos de la clase 10, muestra aleatoria con semilla 42): `MedInc` (ingreso mediano, en decenas de miles de \$), `HouseAge`, `AveRooms` y `AveBedrms` (habitaciones y dormitorios por vivienda), `Population`, `AveOccup` (ocupantes por vivienda), `Latitude`, `Longitude` y `MedHouseVal` (**valor mediano** de la vivienda en unidades de 100 000 \$, **censurado** en 5,00001). Son datos **observacionales** y agregados por distrito.

**Pregunta de negocio.** Una agencia quiere saber cuánto se asocia el ingreso del distrito con el valor de la vivienda y qué otras características importan.

**Ruta.** Auditoría con **filtros razonados** (`AveRooms < 10` y `AveOccup < 6`), censura (clases 2-3 y 10) → regresión múltiple con **HC3** por heterocedasticidad y VIF (clases 10 y 12) → efecto de `MedInc` con IC.
**Trampas:** la **censura** en 5,00001; **AveRooms** cambia de signo al pasar del modelo simple al múltiple; el ajuste no es una predicción (clase 12); y es un dato **ecológico** (distritos, no personas).

### Caso C · Paso 1 — Auditoría, filtros y censura

Sobre `cali` calcula: `n_bruto` (filas), `n_faltantes_c` (total de ausentes), `n_duplicados_c` (filas duplicadas) y `n_censurados_bruto` (distritos con `MedHouseVal >= 5`). Después crea `cali_f`, con `.copy()`, quedándote con las filas de `AveRooms < 10` **y** `AveOccup < 6` (valores extremos que casi seguro son errores o distritos atípicos), y calcula `n_filtrado`, `n_censurados` (censurados que quedan en `cali_f`) y `r_rooms` (correlación de `AveRooms` con `MedHouseVal` en `cali_f`).

In [ ]:
n_bruto = None            # TODO
n_faltantes_c = None      # TODO
n_duplicados_c = None     # TODO
n_censurados_bruto = None # TODO

cali_f = None             # TODO: filtra con la condición conjunta y usa .copy()
n_filtrado = None         # TODO
n_censurados = None       # TODO
r_rooms = None            # TODO

# Comprobación automática
assert all(v is not None for v in (n_bruto, n_faltantes_c, n_duplicados_c, n_censurados_bruto, cali_f, n_filtrado, n_censurados, r_rooms)), "Completa las líneas marcadas con TODO"
assert n_bruto == len(cali) and n_faltantes_c == int(cali.isna().sum().sum()) and n_duplicados_c == int(cali.duplicated().sum()), "len(cali), cali.isna().sum().sum(), cali.duplicated().sum()"
assert n_censurados_bruto == int((cali["MedHouseVal"] >= 5).sum()), "distritos con MedHouseVal >= 5"
_f = cali[(cali["AveRooms"] < 10) & (cali["AveOccup"] < 6)]
assert n_filtrado == len(_f) and len(cali_f) == len(_f), "cali[(cali['AveRooms'] < 10) & (cali['AveOccup'] < 6)]: cuidado con los paréntesis y con & (no 'and')"
assert n_censurados == int((_f["MedHouseVal"] >= 5).sum()), "censurados que quedan en cali_f"
assert np.isclose(r_rooms, _f["AveRooms"].corr(_f["MedHouseVal"])), "correlación en cali_f, no en cali"
print(f"Filas: {n_bruto} → {n_filtrado} tras los filtros; censurados: {n_censurados_bruto} → {n_censurados}")
print(f"Correlación simple de AveRooms con el valor: {r_rooms:.3f}")
print("Caso C, paso 1 correcto ✔")

### Caso C · Paso 2 — Modelo múltiple, errores robustos y colinealidad

Con `cali_f` ajusta la regresión `MedHouseVal ~ MedInc + HouseAge + AveRooms + AveBedrms + AveOccup` de dos maneras: `modelo_c_ols` (errores clásicos) y `modelo_c` (errores **HC3**). Calcula `cociente_se` (error estándar HC3 de `MedInc` dividido por el clásico: ¿cuánto se subestimaba la incertidumbre?), `vif_max_c` (mayor VIF de las cinco explicativas, con `vif_de`) y `signo_cambia` (`True` si el coeficiente de `AveRooms` en `modelo_c` tiene **signo distinto** a `r_rooms`).

In [ ]:
formula_c = "MedHouseVal ~ MedInc + HouseAge + AveRooms + AveBedrms + AveOccup"

modelo_c_ols = None    # TODO
modelo_c = None        # TODO
cociente_se = None     # TODO
vif_max_c = None       # TODO
signo_cambia = None    # TODO

# Comprobación automática
assert all(v is not None for v in (modelo_c_ols, modelo_c, cociente_se, vif_max_c, signo_cambia)), "Completa las líneas marcadas con TODO"
_o = smf.ols(formula_c, data=cali_f).fit()
assert np.allclose(modelo_c_ols.params, _o.params) and np.allclose(modelo_c_ols.bse, _o.bse), "modelo_c_ols: smf.ols(formula_c, data=cali_f).fit()"
_h = smf.ols(formula_c, data=cali_f).fit(cov_type="HC3")
assert np.allclose(modelo_c.bse, _h.bse), "modelo_c debe usar .fit(cov_type='HC3')"
assert np.isclose(cociente_se, _h.bse["MedInc"] / _o.bse["MedInc"]), "cociente_se = modelo_c.bse['MedInc'] / modelo_c_ols.bse['MedInc']"
_v = np.diag(np.linalg.inv(np.corrcoef(cali_f[["MedInc", "HouseAge", "AveRooms", "AveBedrms", "AveOccup"]].values.T))).max()
assert np.isclose(vif_max_c, _v, rtol=1e-6), "vif_max_c = max(vif_de(cali_f[[...cinco explicativas...]]).values())"
assert signo_cambia == bool(np.sign(_h.params["AveRooms"]) != np.sign(cali_f["AveRooms"].corr(cali_f["MedHouseVal"]))), "compara np.sign(coeficiente de AveRooms) con np.sign(r_rooms)"
print(f"El error estándar de MedInc crece un factor {cociente_se:.2f} con HC3; VIF máximo = {vif_max_c:.2f}; ¿AveRooms cambia de signo? {signo_cambia}")
print("Caso C, paso 2 correcto ✔")

### Caso C · Paso 3 — Resultado con incertidumbre

Construye `resultado_c` con las claves: `n` (observaciones de `modelo_c`), `coef_medinc` (coeficiente de `MedInc`: cambio esperado del valor, en unidades de 100 000 $, por cada unidad de `MedInc`, a igualdad de lo demás), `ic_inf` e `ic_sup` (IC 95 % con HC3), `r2` (R² de `modelo_c_ols`), `n_censurados` (los de `cali_f`) y `causal` (`True` o `False`: ¿permiten estos datos afirmar que subir el ingreso de un distrito **causará** ese aumento de valor?).

In [ ]:
resultado_c = {
    "n": None,             # TODO
    "coef_medinc": None,   # TODO
    "ic_inf": None,        # TODO
    "ic_sup": None,        # TODO
    "r2": None,            # TODO
    "n_censurados": None,  # TODO
    "causal": None,        # TODO
}

# Comprobación automática
assert all(v is not None for v in resultado_c.values()), "Completa las líneas marcadas con TODO"
_r = resultado_c
assert _r["n"] == len(cali_f), "n = número de observaciones del modelo (modelo_c.nobs)"
assert np.isclose(_r["coef_medinc"], modelo_c.params["MedInc"]), "coef_medinc = modelo_c.params['MedInc']"
_ic = modelo_c.conf_int().loc["MedInc"]
assert np.isclose(_r["ic_inf"], _ic[0]) and np.isclose(_r["ic_sup"], _ic[1]), "IC 95 % con HC3: modelo_c.conf_int().loc['MedInc']"
assert _r["ic_inf"] < _r["coef_medinc"] < _r["ic_sup"], "el IC debe contener la estimación"
assert np.isclose(_r["r2"], modelo_c_ols.rsquared) and 0 < _r["r2"] < 1, "r2 = modelo_c_ols.rsquared"
assert _r["n_censurados"] == int((cali_f["MedHouseVal"] >= 5).sum()), "censurados de cali_f"
assert _r["causal"] is False, "datos observacionales y agregados: asociación, no causa"
print(f"MedInc: {_r['coef_medinc']:.3f} por unidad (IC 95 % HC3: {_r['ic_inf']:.3f} a {_r['ic_sup']:.3f}); R² = {_r['r2']:.3f}; censurados = {_r['n_censurados']}")
print("Caso C, paso 3 correcto ✔")

---
# Reto final — El informe de una página

Escribe el informe del caso que hayas elegido. Asigna a `caso_elegido` la letra (`"A"`, `"B"` o `"C"`) y a `informe_final` una cadena con **seis secciones**, en este orden: **Pregunta y decisión · Datos · Método · Resultados · Límites · Recomendación** (usa esos títulos). Debe citar las **cifras clave** de tu `resultado_x` (tamaño de muestra, estimación e intervalo), declarar los **límites** y no pasar de una página (unos 3 500 caracteres). Usa la plantilla del capítulo (sección 3). La comprobación automática solo verifica lo mínimo; la calidad la valoras tú con la rúbrica final.

In [ ]:
caso_elegido = None    # TODO: "A", "B" o "C"
informe_final = None   # TODO: una cadena con las seis secciones (puedes usar f-strings con tus resultado_x)

# Comprobación automática
assert caso_elegido in ("A", "B", "C") and isinstance(informe_final, str), "Completa las líneas marcadas con TODO"
_res, _claves = {"A": (resultado_a, ("coef_bmi", "ic_inf", "ic_sup")),
                 "B": (resultado_b, ("dif_pp", "ic_inf_pp", "ic_sup_pp")),
                 "C": (resultado_c, ("coef_medinc", "ic_inf", "ic_sup"))}[caso_elegido]
assert all(v is not None for v in _res.values()), "completa antes los tres pasos de tu caso"

def _norm(t):
    return unicodedata.normalize("NFD", t.lower()).encode("ascii", "ignore").decode()

_texto = _norm(informe_final)
_titulos = ["pregunta", "datos", "metodo", "resultados", "limites", "recomendacion"]
_faltan = [t for t in _titulos if t not in _texto]
assert not _faltan, f"faltan secciones (usa los títulos de la plantilla): {_faltan}"
assert 600 <= len(informe_final) <= 5000, "el informe debe tener entre 600 y 5 000 caracteres (una página)"
assert str(_res["n"]) in re.sub(r"[\s.,]", "", informe_final), "cita el tamaño de muestra n"
_numeros = [float(x.replace(",", ".")) for x in re.findall(r"-?\d+(?:[.,]\d+)?", informe_final)]
for _k in _claves:
    _v = _res[_k]
    assert any(abs(x - _v) <= max(0.06, 0.01 * abs(_v)) for x in _numeros), f"cita la cifra de '{_k}' ({_v:.2f}, redondeada)"
print(f"Informe con las seis secciones, los límites y las cifras del caso {caso_elegido} citadas.")
print("Reto final correcto ✔")

---
## Cierre de la práctica y del curso

Has recorrido un análisis completo, de la pregunta al informe. Con esto **termina el curso**. Para seguir, el paso natural es un curso de Machine Learning, que retoma la validación y los modelos de las clases 11 y 12 con herramientas de predicción; y la documentación de `statsmodels` y `scipy` para profundizar en cada método.

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un paso. Ejecútalas en orden si quieres comparar con tu resultado. La solución del reto final redacta un informe de ejemplo para cada caso (aquí se muestra el del caso B).

### Solución del caso A · 1

In [ ]:
n_a = len(diab)
n_faltantes_a = int(diab.isna().sum().sum())
n_duplicados_a = int(diab.duplicated().sum())
r_bmi = diab["bmi"].corr(diab["target"])
var_max_corr = diab.drop(columns="target").corrwith(diab["target"]).abs().idxmax()
r_s1_s2 = diab["s1"].corr(diab["s2"])

# Comprobación automática
assert all(v is not None for v in (n_a, n_faltantes_a, n_duplicados_a, r_bmi, var_max_corr, r_s1_s2)), "Completa las líneas marcadas con TODO"
assert n_a == len(diab), "n_a = len(diab)"
assert n_faltantes_a == int(diab.isna().sum().sum()), "suma de todos los ausentes: diab.isna().sum().sum()"
assert n_duplicados_a == int(diab.duplicated().sum()), "diab.duplicated().sum()"
assert np.isclose(r_bmi, diab["bmi"].corr(diab["target"])), "r de Pearson entre bmi y target"
assert var_max_corr == diab.drop(columns="target").corrwith(diab["target"]).abs().idxmax(), "usa el valor absoluto de la correlación y idxmax()"
assert np.isclose(r_s1_s2, diab["s1"].corr(diab["s2"])), "r de Pearson entre s1 y s2"
print(f"n = {n_a}, ausentes = {n_faltantes_a}, duplicados = {n_duplicados_a}, r(bmi, target) = {r_bmi:.3f}, mayor |r|: {var_max_corr}, r(s1, s2) = {r_s1_s2:.3f}")
print("Caso A, paso 1 correcto ✔")

### Solución del caso A · 2

In [ ]:
X_a = diab.drop(columns="target").copy()
X_a["sex"] = (X_a["sex"] == 2).astype(float)     # 0/1: solo para poder calcular el VIF

vif_a = vif_de(X_a)
vif_max_a = max(vif_a.values())
variables_vif_alto = sorted(c for c, v in vif_a.items() if v > 10)
modelo_a = smf.ols("target ~ age + C(sex) + bmi + bp + s3 + s5 + s6", data=diab).fit(cov_type="HC3")

# Comprobación automática
assert all(v is not None for v in (vif_a, vif_max_a, variables_vif_alto, modelo_a)), "Completa las líneas marcadas con TODO"
_Xn = X_a.astype(float).values
_ref = dict(zip(X_a.columns, np.diag(np.linalg.inv(np.corrcoef(_Xn.T)))))      # VIF = diagonal de la inversa de la matriz de correlaciones
assert set(vif_a) == set(X_a.columns) and all(np.isclose(vif_a[c], _ref[c], rtol=1e-6) for c in _ref), "vif_a = vif_de(X_a): un VIF por cada columna de X_a"
assert np.isclose(vif_max_a, max(_ref.values())), "vif_max_a = max(vif_a.values())"
assert variables_vif_alto == sorted(c for c, v in _ref.items() if v > 10), "sorted([c for c, v in vif_a.items() if v > 10])"
assert getattr(modelo_a, "cov_type", None) == "HC3", "falta .fit(cov_type='HC3')"
assert set(modelo_a.params.index) == {"Intercept", "C(sex)[T.2.0]", "age", "bmi", "bp", "s3", "s5", "s6"}, "fórmula: target ~ age + C(sex) + bmi + bp + s3 + s5 + s6"
print(f"VIF máximo (modelo completo) = {vif_max_a:.1f}; variables con VIF > 10:", variables_vif_alto)
print("Coeficiente de bmi (HC3):", round(modelo_a.params["bmi"], 3))
print("Caso A, paso 2 correcto ✔")

### Solución del caso A · 3

In [ ]:
X_red = X_a[["age", "sex", "bmi", "bp", "s3", "s5", "s6"]]

ic = modelo_a.conf_int().loc["bmi"]
resultado_a = {
    "n": int(modelo_a.nobs),
    "coef_bmi": modelo_a.params["bmi"],
    "ic_inf": ic[0],
    "ic_sup": ic[1],
    "r2_adj": smf.ols("target ~ age + C(sex) + bmi + bp + s3 + s5 + s6", data=diab).fit().rsquared_adj,
    "vif_max": max(vif_de(X_red).values()),
    "causal": False,
}

# Comprobación automática
assert all(v is not None for v in resultado_a.values()), "Completa las líneas marcadas con TODO"
_r = resultado_a
assert _r["n"] == len(diab), "n = número de observaciones del modelo (modelo_a.nobs)"
assert np.isclose(_r["coef_bmi"], modelo_a.params["bmi"]), "coef_bmi = modelo_a.params['bmi']"
_ic = modelo_a.conf_int().loc["bmi"]
assert np.isclose(_r["ic_inf"], _ic[0]) and np.isclose(_r["ic_sup"], _ic[1]), "IC 95 %: modelo_a.conf_int().loc['bmi']"
assert _r["ic_inf"] < _r["coef_bmi"] < _r["ic_sup"], "el IC debe contener la estimación"
assert np.isclose(_r["r2_adj"], smf.ols("target ~ age + C(sex) + bmi + bp + s3 + s5 + s6", data=diab).fit().rsquared_adj), "R² corregido: .rsquared_adj (el del modelo sin HC3 es el mismo valor)"
_v = np.diag(np.linalg.inv(np.corrcoef(X_red.astype(float).values.T))).max()
assert np.isclose(_r["vif_max"], _v, rtol=1e-6), "vif_max = max(vif_de(X_red).values())"
assert _r["causal"] is False, "son datos observacionales: no se puede afirmar causa"
print(f"Efecto de bmi: {_r['coef_bmi']:.2f} por unidad (IC 95 % HC3: {_r['ic_inf']:.2f} a {_r['ic_sup']:.2f}); R² corregido = {_r['r2_adj']:.3f}; VIF máximo = {_r['vif_max']:.2f}")
print("Caso A, paso 3 correcto ✔")

### Solución del caso B · 1

In [ ]:
n_b = len(ecom)
n_faltantes_b = int(ecom.isna().sum().sum())
n_duplicados_id = int(ecom["id_usuario"].duplicated().sum())
tasa_A = ecom.loc[ecom["grupo"] == "A", "convirtio"].mean() * 100
tasa_B = ecom.loc[ecom["grupo"] == "B", "convirtio"].mean() * 100
asimetria_ingresos = stats.skew(ecom.loc[ecom["convirtio"] == 1, "ingresos"])

# Comprobación automática
assert all(v is not None for v in (n_b, n_faltantes_b, n_duplicados_id, tasa_A, tasa_B, asimetria_ingresos)), "Completa las líneas marcadas con TODO"
assert n_b == len(ecom), "n_b = len(ecom)"
assert n_faltantes_b == int(ecom.isna().sum().sum()) and n_duplicados_id == int(ecom["id_usuario"].duplicated().sum()), "ecom.isna().sum().sum() y ecom['id_usuario'].duplicated().sum()"
assert np.isclose(tasa_A, ecom.loc[ecom["grupo"] == "A", "convirtio"].mean() * 100), "tasa_A: media de convirtio en el grupo A, multiplicada por 100"
assert np.isclose(tasa_B, ecom.loc[ecom["grupo"] == "B", "convirtio"].mean() * 100), "tasa_B: media de convirtio en el grupo B, multiplicada por 100"
assert np.isclose(asimetria_ingresos, stats.skew(ecom.loc[ecom["convirtio"] == 1, "ingresos"])), "stats.skew de ingresos filtrando convirtio == 1"
assert asimetria_ingresos > 1, "una asimetría > 1 indica cola larga: mejor la mediana o el bootstrap que la media"
print(f"n = {n_b}, ausentes = {n_faltantes_b}, ids repetidos = {n_duplicados_id}, conversión A = {tasa_A:.2f} %, conversión B = {tasa_B:.2f} %, asimetría de ingresos = {asimetria_ingresos:.2f}")
print("Caso B, paso 1 correcto ✔")

### Solución del caso B · 2

In [ ]:
conv = ecom.groupby("grupo")["convirtio"].agg(["sum", "count"])     # conversiones y usuarios por grupo

tamanos_b = {"A": int(conv.loc["A", "count"]), "B": int(conv.loc["B", "count"])}
chi2_srm, p_srm = stats.chisquare([tamanos_b["A"], tamanos_b["B"]])
srm_detectado = bool(p_srm < 0.001)
p_dispositivos = stats.chi2_contingency(pd.crosstab(ecom["dispositivo"], ecom["grupo"]))[1]
dif_pp = (conv.loc["B", "sum"] / conv.loc["B", "count"] - conv.loc["A", "sum"] / conv.loc["A", "count"]) * 100
lo, hi = confint_proportions_2indep(conv.loc["B", "sum"], conv.loc["B", "count"], conv.loc["A", "sum"], conv.loc["A", "count"], method="newcomb")
ic_inf_pp, ic_sup_pp = lo * 100, hi * 100
p_valor = proportions_ztest([conv.loc["B", "sum"], conv.loc["A", "sum"]], [conv.loc["B", "count"], conv.loc["A", "count"]])[1]

# Comprobación automática
assert all(v is not None for v in (tamanos_b, chi2_srm, p_srm, srm_detectado, p_dispositivos, dif_pp, ic_inf_pp, ic_sup_pp, p_valor)), "Completa las líneas marcadas con TODO"
assert tamanos_b == {"A": int(conv.loc["A", "count"]), "B": int(conv.loc["B", "count"])}, "tamanos_b = {'A': usuarios de A, 'B': usuarios de B}"
_c2, _p = stats.chisquare([conv.loc["A", "count"], conv.loc["B", "count"]])
assert np.isclose(chi2_srm, _c2) and np.isclose(p_srm, _p), "stats.chisquare([n_A, n_B]) reparte por defecto a partes iguales"
assert srm_detectado == bool(_p < 0.001), "srm_detectado = p_srm < 0.001"
assert np.isclose(p_dispositivos, stats.chi2_contingency(pd.crosstab(ecom["dispositivo"], ecom["grupo"]))[1]), "chi2_contingency(pd.crosstab(dispositivo, grupo))[1]"
_d = (conv.loc["B", "sum"] / conv.loc["B", "count"] - conv.loc["A", "sum"] / conv.loc["A", "count"]) * 100
assert np.isclose(dif_pp, _d), "dif_pp = (tasa de B − tasa de A) · 100"
_lo, _hi = confint_proportions_2indep(conv.loc["B", "sum"], conv.loc["B", "count"], conv.loc["A", "sum"], conv.loc["A", "count"], method="newcomb")
assert np.isclose(ic_inf_pp, _lo * 100) and np.isclose(ic_sup_pp, _hi * 100), "confint_proportions_2indep(conv_B, n_B, conv_A, n_A, method='newcomb'), multiplicado por 100"
assert ic_inf_pp < dif_pp < ic_sup_pp, "el IC debe contener la estimación"
assert np.isclose(p_valor, proportions_ztest([conv.loc["B", "sum"], conv.loc["A", "sum"]], [conv.loc["B", "count"], conv.loc["A", "count"]])[1]), "proportions_ztest([conv_B, conv_A], [n_B, n_A])[1]"
print(f"Usuarios: {tamanos_b}  SRM: chi² = {chi2_srm:.1f}, p = {p_srm:.2g}  (¡alarma!)" if srm_detectado else f"SRM: p = {p_srm:.3f}")
print(f"Dispositivo × grupo: p = {p_dispositivos:.2g}")
print(f"Diferencia B − A = {dif_pp:.2f} pp (IC 95 %: {ic_inf_pp:.2f} a {ic_sup_pp:.2f}), p = {p_valor:.4f}")
print("Caso B, paso 2 correcto ✔")

### Solución del caso B · 3

In [ ]:
p_seg = []
for _, seg in ecom.groupby(["dispositivo", "canal"]):
    t = seg.groupby("grupo")["convirtio"].agg(["sum", "count"])
    p_seg.append(proportions_ztest([t.loc["B", "sum"], t.loc["A", "sum"]], [t.loc["B", "count"], t.loc["A", "count"]])[1])

p_holm = multipletests(p_seg, method="holm")[1]
n_sig_sin = int(np.sum(np.array(p_seg) < 0.05))
n_sig_holm = int(np.sum(p_holm < 0.05))

if srm_detectado:
    decision_b = "revisar_asignacion"
elif ic_inf_pp > 0:
    decision_b = "lanzar_B"
else:
    decision_b = "no_lanzar_B"

resultado_b = {
    "n": n_b,
    "dif_pp": dif_pp,
    "ic_inf_pp": ic_inf_pp,
    "ic_sup_pp": ic_sup_pp,
    "srm_p": p_srm,
    "n_seg_sig_holm": n_sig_holm,
    "decision": decision_b,
}

# Comprobación automática
assert p_holm is not None and n_sig_sin is not None and n_sig_holm is not None and all(v is not None for v in resultado_b.values()), "Completa las líneas marcadas con TODO"
_ps = []
for _, _s in ecom.groupby(["dispositivo", "canal"]):
    _t = _s.groupby("grupo")["convirtio"].agg(["sum", "count"])
    _ps.append(proportions_ztest([_t.loc["B", "sum"], _t.loc["A", "sum"]], [_t.loc["B", "count"], _t.loc["A", "count"]])[1])
assert len(p_seg) == 6 and np.allclose(p_seg, _ps), "6 p-values, uno por segmento, con B primero en proportions_ztest"
assert np.allclose(p_holm, multipletests(_ps, method="holm")[1]), "multipletests(p_seg, method='holm')[1]"
assert n_sig_sin == int(np.sum(np.array(_ps) < 0.05)) and n_sig_holm == int(np.sum(multipletests(_ps, method="holm")[1] < 0.05)), "cuenta cuántos p-values (sin corregir y corregidos) son < 0.05"
assert n_sig_holm <= n_sig_sin, "corregir nunca puede aumentar el número de significativos"
_r = resultado_b
assert _r["n"] == len(ecom) and np.isclose(_r["dif_pp"], dif_pp) and np.isclose(_r["ic_inf_pp"], ic_inf_pp) and np.isclose(_r["ic_sup_pp"], ic_sup_pp), "usa las cifras del paso 2"
assert np.isclose(_r["srm_p"], p_srm) and _r["n_seg_sig_holm"] == n_sig_holm, "srm_p = p_srm y n_seg_sig_holm = n_sig_holm"
assert _r["ic_inf_pp"] < _r["dif_pp"] < _r["ic_sup_pp"], "el IC debe contener la estimación"
assert _r["decision"] == ("revisar_asignacion" if p_srm < 0.001 else ("lanzar_B" if ic_inf_pp > 0 else "no_lanzar_B")), "si hay SRM la decisión es revisar la asignación, aunque el p-value sea pequeño"
print(f"Segmentos significativos sin corregir: {n_sig_sin}; con Holm: {n_sig_holm}")
print(f"Decisión: {_r['decision']}")
print("Caso B, paso 3 correcto ✔")

### Solución del caso C · 1

In [ ]:
n_bruto = len(cali)
n_faltantes_c = int(cali.isna().sum().sum())
n_duplicados_c = int(cali.duplicated().sum())
n_censurados_bruto = int((cali["MedHouseVal"] >= 5).sum())

cali_f = cali[(cali["AveRooms"] < 10) & (cali["AveOccup"] < 6)].copy()
n_filtrado = len(cali_f)
n_censurados = int((cali_f["MedHouseVal"] >= 5).sum())
r_rooms = cali_f["AveRooms"].corr(cali_f["MedHouseVal"])

# Comprobación automática
assert all(v is not None for v in (n_bruto, n_faltantes_c, n_duplicados_c, n_censurados_bruto, cali_f, n_filtrado, n_censurados, r_rooms)), "Completa las líneas marcadas con TODO"
assert n_bruto == len(cali) and n_faltantes_c == int(cali.isna().sum().sum()) and n_duplicados_c == int(cali.duplicated().sum()), "len(cali), cali.isna().sum().sum(), cali.duplicated().sum()"
assert n_censurados_bruto == int((cali["MedHouseVal"] >= 5).sum()), "distritos con MedHouseVal >= 5"
_f = cali[(cali["AveRooms"] < 10) & (cali["AveOccup"] < 6)]
assert n_filtrado == len(_f) and len(cali_f) == len(_f), "cali[(cali['AveRooms'] < 10) & (cali['AveOccup'] < 6)]: cuidado con los paréntesis y con & (no 'and')"
assert n_censurados == int((_f["MedHouseVal"] >= 5).sum()), "censurados que quedan en cali_f"
assert np.isclose(r_rooms, _f["AveRooms"].corr(_f["MedHouseVal"])), "correlación en cali_f, no en cali"
print(f"Filas: {n_bruto} → {n_filtrado} tras los filtros; censurados: {n_censurados_bruto} → {n_censurados}")
print(f"Correlación simple de AveRooms con el valor: {r_rooms:.3f}")
print("Caso C, paso 1 correcto ✔")

### Solución del caso C · 2

In [ ]:
formula_c = "MedHouseVal ~ MedInc + HouseAge + AveRooms + AveBedrms + AveOccup"

modelo_c_ols = smf.ols(formula_c, data=cali_f).fit()
modelo_c = smf.ols(formula_c, data=cali_f).fit(cov_type="HC3")
cociente_se = modelo_c.bse["MedInc"] / modelo_c_ols.bse["MedInc"]
vif_max_c = max(vif_de(cali_f[["MedInc", "HouseAge", "AveRooms", "AveBedrms", "AveOccup"]]).values())
signo_cambia = bool(np.sign(modelo_c.params["AveRooms"]) != np.sign(r_rooms))

# Comprobación automática
assert all(v is not None for v in (modelo_c_ols, modelo_c, cociente_se, vif_max_c, signo_cambia)), "Completa las líneas marcadas con TODO"
_o = smf.ols(formula_c, data=cali_f).fit()
assert np.allclose(modelo_c_ols.params, _o.params) and np.allclose(modelo_c_ols.bse, _o.bse), "modelo_c_ols: smf.ols(formula_c, data=cali_f).fit()"
_h = smf.ols(formula_c, data=cali_f).fit(cov_type="HC3")
assert np.allclose(modelo_c.bse, _h.bse), "modelo_c debe usar .fit(cov_type='HC3')"
assert np.isclose(cociente_se, _h.bse["MedInc"] / _o.bse["MedInc"]), "cociente_se = modelo_c.bse['MedInc'] / modelo_c_ols.bse['MedInc']"
_v = np.diag(np.linalg.inv(np.corrcoef(cali_f[["MedInc", "HouseAge", "AveRooms", "AveBedrms", "AveOccup"]].values.T))).max()
assert np.isclose(vif_max_c, _v, rtol=1e-6), "vif_max_c = max(vif_de(cali_f[[...cinco explicativas...]]).values())"
assert signo_cambia == bool(np.sign(_h.params["AveRooms"]) != np.sign(cali_f["AveRooms"].corr(cali_f["MedHouseVal"]))), "compara np.sign(coeficiente de AveRooms) con np.sign(r_rooms)"
print(f"El error estándar de MedInc crece un factor {cociente_se:.2f} con HC3; VIF máximo = {vif_max_c:.2f}; ¿AveRooms cambia de signo? {signo_cambia}")
print("Caso C, paso 2 correcto ✔")

### Solución del caso C · 3

In [ ]:
ic = modelo_c.conf_int().loc["MedInc"]
resultado_c = {
    "n": int(modelo_c.nobs),
    "coef_medinc": modelo_c.params["MedInc"],
    "ic_inf": ic[0],
    "ic_sup": ic[1],
    "r2": modelo_c_ols.rsquared,
    "n_censurados": n_censurados,
    "causal": False,
}

# Comprobación automática
assert all(v is not None for v in resultado_c.values()), "Completa las líneas marcadas con TODO"
_r = resultado_c
assert _r["n"] == len(cali_f), "n = número de observaciones del modelo (modelo_c.nobs)"
assert np.isclose(_r["coef_medinc"], modelo_c.params["MedInc"]), "coef_medinc = modelo_c.params['MedInc']"
_ic = modelo_c.conf_int().loc["MedInc"]
assert np.isclose(_r["ic_inf"], _ic[0]) and np.isclose(_r["ic_sup"], _ic[1]), "IC 95 % con HC3: modelo_c.conf_int().loc['MedInc']"
assert _r["ic_inf"] < _r["coef_medinc"] < _r["ic_sup"], "el IC debe contener la estimación"
assert np.isclose(_r["r2"], modelo_c_ols.rsquared) and 0 < _r["r2"] < 1, "r2 = modelo_c_ols.rsquared"
assert _r["n_censurados"] == int((cali_f["MedHouseVal"] >= 5).sum()), "censurados de cali_f"
assert _r["causal"] is False, "datos observacionales y agregados: asociación, no causa"
print(f"MedInc: {_r['coef_medinc']:.3f} por unidad (IC 95 % HC3: {_r['ic_inf']:.3f} a {_r['ic_sup']:.3f}); R² = {_r['r2']:.3f}; censurados = {_r['n_censurados']}")
print("Caso C, paso 3 correcto ✔")

### Solución del reto final

In [ ]:
def num(x, d=2):
    return f"{x:.{d}f}".replace(".", ",")

textos = {
    "A": f"""### Informe — Progresión de la diabetes

**Pregunta y decisión.** ¿Qué características basales se asocian con más progresión de la enfermedad? Decisión: a quién priorizar en el seguimiento.

**Datos.** {resultado_a["n"]} pacientes, sin ausentes ni duplicados; datos observacionales. `s1`, `s2` y `s3` son casi redundantes (VIF de hasta {num(vif_max_a, 0)}).

**Método.** Regresión lineal múltiple sin las variables redundantes (VIF máximo {num(resultado_a["vif_max"])} en el modelo final) y errores estándar robustos HC3.

**Resultados.** Por cada unidad más de `bmi` la progresión esperada sube {num(resultado_a["coef_bmi"])} puntos (IC 95 %: {num(resultado_a["ic_inf"])} a {num(resultado_a["ic_sup"])}), a igualdad de las demás variables. El modelo explica cerca de la mitad de la variación (R² corregido = {num(resultado_a["r2_adj"], 3)}).

**Límites.** Datos observacionales de {resultado_a["n"]} pacientes: es asociación, no causa. La elección de variables se hizo con los mismos datos (los IC son algo optimistas) y no se validó con datos nuevos. No se interpreta el signo de `sex` porque su codificación no está documentada.

**Recomendación.** Usar el `bmi` y la presión como señales para priorizar el seguimiento, y validar el modelo con datos nuevos antes de usarlo para predecir.
""",
    "B": f"""### Informe — Test A/B de la nueva página de pago

**Pregunta y decisión.** ¿Aumenta la versión B la conversión? Decisión: lanzarla a todos los usuarios o no.

**Datos.** {resultado_b["n"]} usuarios (sintéticos), sin ausentes ni ids repetidos. Reparto A/B: {tamanos_b["A"]} / {tamanos_b["B"]}.

**Método.** Control SRM (chi-cuadrado), equilibrio de dispositivos, test de dos proporciones con IC de la diferencia y seis segmentos dispositivo × canal con corrección de Holm.

**Resultados.** Conversión A = {num(tasa_A)} % y B = {num(tasa_B)} %: diferencia de {num(resultado_b["dif_pp"])} pp (IC 95 %: {num(resultado_b["ic_inf_pp"])} a {num(resultado_b["ic_sup_pp"])}). Sin corregir, {n_sig_sin} segmentos salen significativos; con Holm, {resultado_b["n_seg_sig_holm"]}. Pero el reparto tiene **SRM** (p = {p_srm:.1e}) y los dispositivos no están equilibrados entre grupos (p = {p_dispositivos:.3f}).

**Límites.** Con SRM, la aleatorización no es fiable y la diferencia observada puede estar sesgada (más móvil en B, donde el efecto es distinto). Los segmentos tienen poca potencia. Los ingresos, con cola larga, no se analizaron con medias.

**Recomendación.** No decidir todavía: investigar la causa del SRM, corregirla y repetir o reanalizar (estratificando por dispositivo). Si se confirma, el efecto plausible es positivo pero modesto.
""",
    "C": f"""### Informe — Valor de la vivienda en California

**Pregunta y decisión.** ¿Cuánto se asocia el ingreso del distrito con el valor de la vivienda? Decisión: qué variables usar para orientar la valoración de zonas.

**Datos.** {n_bruto} distritos; se excluyen {n_bruto - n_filtrado} con `AveRooms` ≥ 10 o `AveOccup` ≥ 6 (valores extremos). Quedan {resultado_c["n"]}, de los cuales {resultado_c["n_censurados"]} tienen el valor censurado en 5,00001. Sin ausentes ni duplicados.

**Método.** Regresión lineal múltiple con errores robustos HC3 (los clásicos subestimaban la incertidumbre por un factor {num(cociente_se)}); VIF máximo {num(vif_max_c)}.

**Resultados.** Por cada unidad más de ingreso mediano (10 000 $), el valor mediano esperado sube {num(resultado_c["coef_medinc"], 3)} (unidades de 100 000 $; IC 95 %: {num(resultado_c["ic_inf"], 3)} a {num(resultado_c["ic_sup"], 3)}), a igualdad del resto. R² = {num(resultado_c["r2"], 3)}. `AveRooms` cambia de signo entre el modelo simple (r = {num(r_rooms)}) y el múltiple.

**Límites.** Datos observacionales y agregados por distrito (falacia ecológica): asociación, no causa. La censura en 5,00001 subestima los distritos más caros; no se usaron latitud ni longitud, así que falta el efecto de la ubicación.

**Recomendación.** Usar el ingreso como primera señal de valor, con cautela en los distritos de valor muy alto, e incorporar la ubicación antes de tomar decisiones por zona.
""",
}
caso_elegido = "B"
informe_final = textos[caso_elegido]
print(informe_final)

assert caso_elegido in ("A", "B", "C") and isinstance(informe_final, str), "Completa las líneas marcadas con TODO"
_res, _claves = {"A": (resultado_a, ("coef_bmi", "ic_inf", "ic_sup")),
                 "B": (resultado_b, ("dif_pp", "ic_inf_pp", "ic_sup_pp")),
                 "C": (resultado_c, ("coef_medinc", "ic_inf", "ic_sup"))}[caso_elegido]
assert all(v is not None for v in _res.values()), "completa antes los tres pasos de tu caso"

def _norm(t):
    return unicodedata.normalize("NFD", t.lower()).encode("ascii", "ignore").decode()

_texto = _norm(informe_final)
_titulos = ["pregunta", "datos", "metodo", "resultados", "limites", "recomendacion"]
_faltan = [t for t in _titulos if t not in _texto]
assert not _faltan, f"faltan secciones (usa los títulos de la plantilla): {_faltan}"
assert 600 <= len(informe_final) <= 5000, "el informe debe tener entre 600 y 5 000 caracteres (una página)"
assert str(_res["n"]) in re.sub(r"[\s.,]", "", informe_final), "cita el tamaño de muestra n"
_numeros = [float(x.replace(",", ".")) for x in re.findall(r"-?\d+(?:[.,]\d+)?", informe_final)]
for _k in _claves:
    _v = _res[_k]
    assert any(abs(x - _v) <= max(0.06, 0.01 * abs(_v)) for x in _numeros), f"cita la cifra de '{_k}' ({_v:.2f}, redondeada)"
print(f"Informe con las seis secciones, los límites y las cifras del caso {caso_elegido} citadas.")
print("Reto final correcto ✔")

---
## Autoevaluación con la rúbrica

Puntúa tu proyecto de 0 (no lo hice) a 2 (bien hecho) en cada criterio y anota el total (máximo 20):

| # | Criterio | 0 / 1 / 2 |
|---|---|---|
| 1 | Pregunta de negocio y decisión definidas | |
| 2 | Auditoría de los datos (tipos, ausentes, duplicados, rangos) | |
| 3 | Exploración antes de modelar | |
| 4 | Método justificado (tabla de decisión) | |
| 5 | Supuestos y diagnóstico comprobados | |
| 6 | Efecto + intervalo de confianza, no solo p-value | |
| 7 | Multiplicidad, selección de variables y colinealidad controladas | |
| 8 | Límites y sesgos declarados | |
| 9 | Sin afirmaciones causales indebidas | |
| 10 | Informe claro, breve y reproducible | |

Después anota:

- **¿Qué caso elegí y qué paso me costó más?**
- **¿Qué herramienta de qué clase tendría que repasar?**
- **Tiempo dedicado:** ___ minutos
- **Qué haría distinto con más tiempo o más datos:** ...